<a href="https://colab.research.google.com/github/FatemehNMT/Visual-SLAM-Book-Google-Colab/blob/main/Chapter_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Chapter 9 Filters and Optimization Approaches: Part II**

## **SLAM Book 2: Pose Graph Using g2o Built-in Classes**

**Page 232**\
https://github.com/gaoxiang12/slambook2/ch10/pose_graph_g2o_SE3.cpp

In [ ]:
%cd /content/
# !cp "/content/gdrive/MyDrive/slambook_data/ch10/useSophus" /content

/content


In [ ]:
%cd /content/
!rm -r MyPGExample
!mkdir MyPGExample
%cd MyPGExample

/content
rm: cannot remove 'MyPGExample': No such file or directory
/content/MyPGExample


In [ ]:
%cd /content/MyPGExample

/content/MyPGExample


In [ ]:
%%writefile pose_graph_g2o_SE3.cpp

#include <iostream>
#include <fstream>
#include <string>

#include <g2o/types/slam3d/types_slam3d.h>
#include <g2o/core/block_solver.h>
#include <g2o/core/optimization_algorithm_levenberg.h>
#include <g2o/solvers/eigen/linear_solver_eigen.h>

using namespace std;

/************************************************
* This program demonstrates how to use g2o solver for pose graph optimization
 * sphere.g2o is an artificially generated Pose graph, let's optimize it.
 * Although the entire graph can be read directly through the load function,
 * we still implement the reading code ourselves in order to gain a deeper understanding.
 * SE3 in g2o/types/slam3d/ is used here to represent the pose, which is essentially a
 * quaternion rather than a Lie algebra.
 * **********************************************/

int main(int argc, char **argv) {
    if (argc != 2) {
        cout << "Usage: pose_graph_g2o_SE3 sphere.g2o" << endl;
        return 1;
    }
    ifstream fin(argv[1]);
    if (!fin) {
        cout << "file " << argv[1] << " does not exist." << endl;
        return 1;
    }

// *****************************************************************
 // Build graph optimization, first set up g2o
    typedef g2o::BlockSolver<g2o::BlockSolverTraits<6, 6>> BlockSolverType;
    // pose is 6, landmark is 6
 // The optimization variable dimension of each error term is 6, and the error value dimension is 6

    typedef g2o::LinearSolverEigen<BlockSolverType::PoseMatrixType> LinearSolverType; // Linear solver type
    // Gradient descent method, you can choose from GN, LM, DogLeg
    auto solver = new g2o::OptimizationAlgorithmLevenberg(
        std::make_unique<BlockSolverType>(std::make_unique<LinearSolverType>()));
    g2o::SparseOptimizer optimizer;     // graphical model
    optimizer.setAlgorithm(solver);     // Set up solver
    optimizer.setVerbose(true);         // Turn on debug output

    int vertexCnt = 0, edgeCnt = 0;     // Number of vertices and edges
    while (!fin.eof()) {
        string name;
        fin >> name;
        if (name == "VERTEX_SE3:QUAT") {
            // SE3 顶点
            g2o::VertexSE3 *v = new g2o::VertexSE3();
            int index = 0;
            fin >> index;
            v->setId(index);
            v->read(fin);
            optimizer.addVertex(v);
            vertexCnt++;
            if (index == 0)
                v->setFixed(true);
        } else if (name == "EDGE_SE3:QUAT") {
            // SE3-SE3 side
            g2o::EdgeSE3 *e = new g2o::EdgeSE3();
            int idx1, idx2;     // Two related vertices
            fin >> idx1 >> idx2;
            e->setId(edgeCnt++);
            e->setVertex(0, optimizer.vertices()[idx1]);
            e->setVertex(1, optimizer.vertices()[idx2]);
            e->read(fin);
            optimizer.addEdge(e);
        }
        if (!fin.good()) break;
    }

    cout << "read total " << vertexCnt << " vertices, " << edgeCnt << " edges." << endl;

    cout << "optimizing ..." << endl;
    optimizer.initializeOptimization();
    optimizer.optimize(30);

    cout << "saving optimization results ..." << endl;
    optimizer.save("result.g2o");

    return 0;
}

Overwriting pose_graph_g2o_SE3.cpp


In [ ]:
%%writefile CMakeLists.txt
cmake_minimum_required(VERSION 3.22)
project(vo1)

set(CMAKE_BUILD_TYPE "Release")
set(CMAKE_CXX_FLAGS "-std=c++17 -O2")

# Eigen
find_package(Eigen3 REQUIRED)
include_directories(${EIGEN3_INCLUDE_DIR})

# OpenCV
find_package(OpenCV REQUIRED)
include_directories(${OpenCV_INCLUDE_DIRS})

# Sophus
find_package(Sophus REQUIRED)
include_directories(${Sophus_INCLUDE_DIRS})

# g2o
find_package(g2o REQUIRED)
include_directories(${g2o_INCLUDE_DIRS})

add_executable(pose_graph_g2o_SE3 pose_graph_g2o_SE3.cpp)
target_link_libraries(pose_graph_g2o_SE3
        g2o_core g2o_stuff
        g2o_types_sba g2o_types_slam3d g2o_solver_dense
        Sophus::Sophus
        ${CHOLMOD_LIBRARIES}
        ${OpenCV_LIBS})

Writing CMakeLists.txt


In [ ]:
!cp "/content/drive/MyDrive/slambook_data/ch10/sphere.g2o" /content/MyPGExample

In [ ]:
!ls /content/MyPGExample

CMakeLists.txt	pose_graph_g2o_SE3.cpp	sphere.g2o


In [ ]:
!rm -r myBuild
!mkdir myBuild
%cd myBuild/

rm: cannot remove 'myBuild': No such file or directory
/content/MyPGExample/myBuild


In [ ]:
!cmake ..

-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Found OpenCV: /usr (found version "4.5.4")
-- Found OpenGL: /usr/lib/x86_64-linux-gnu/libOpenGL.so
-- Configuring done (0.5s)
-- Generating done (0.0s)
-- Build files have been written to: /content/MyPGExample/myBuild


In [ ]:
!make pose_graph_g2o_SE3

[ 50%] Building CXX object CMakeFiles/pose_graph_g2o_SE3.dir/pose_graph_g2o_SE3.cpp.o
[100%] Linking CXX executable pose_graph_g2o_SE3
[100%] Built target pose_graph_g2o_SE3


In [ ]:
! ./pose_graph_g2o_SE3

./pose_graph_g2o_SE3: error while loading shared libraries: libg2o_core.so.0.2: cannot open shared object file: No such file or directory


In [ ]:
! LD_LIBRARY_PATH=/usr/local/lib:$LD_LIBRARY_PATH ./pose_graph_g2o_SE3

Usage: pose_graph_g2o_SE3 sphere.g2o


In [ ]:
!pwd

/content/MyPGExample/myBuild


In [ ]:
! LD_LIBRARY_PATH=/usr/local/lib:$LD_LIBRARY_PATH ./pose_graph_g2o_SE3 sphere.g2o

file sphere.g2o does not exist.


In [ ]:
! LD_LIBRARY_PATH=/usr/local/lib:$LD_LIBRARY_PATH ./pose_graph_g2o_SE3 /content/MyPGExample/sphere.g2o

read total 2500 vertices, 9799 edges.
optimizing ...
iteration= 0	 chi2= 1023011093.967641	 time= 0.500261	 cumTime= 0.500261	 edges= 9799	 schur= 0	 lambda= 805.622433	 levenbergIter= 1
iteration= 1	 chi2= 385118688.233187	 time= 0.467084	 cumTime= 0.967345	 edges= 9799	 schur= 0	 lambda= 537.081622	 levenbergIter= 1
iteration= 2	 chi2= 166223726.693657	 time= 0.468638	 cumTime= 1.43598	 edges= 9799	 schur= 0	 lambda= 358.054415	 levenbergIter= 1
iteration= 3	 chi2= 86610874.269315	 time= 0.470461	 cumTime= 1.90644	 edges= 9799	 schur= 0	 lambda= 238.702943	 levenbergIter= 1
iteration= 4	 chi2= 40582782.710189	 time= 0.454575	 cumTime= 2.36102	 edges= 9799	 schur= 0	 lambda= 159.135295	 levenbergIter= 1
iteration= 5	 chi2= 15055383.753040	 time= 0.462012	 cumTime= 2.82303	 edges= 9799	 schur= 0	 lambda= 101.425210	 levenbergIter= 1
iteration= 6	 chi2= 6715193.487654	 time= 0.448516	 cumTime= 3.27155	 edges= 9799	 schur= 0	 lambda= 37.664667	 levenbergIter= 1
iteration= 7	 chi2= 217194

## **SLAM Book 2: Pose Graph Using Sophus**

**Page 234**\
https://github.com/gaoxiang12/slambook2/ch10/pose_graph_g2o_lie_algebra.cpp

In [ ]:
%cd /content/
!mkdir MyLiePGExample
%cd MyLiePGExample

/content
mkdir: cannot create directory ‘MyLiePGExample’: File exists
/content/MyLiePGExample


In [ ]:
%%writefile pose_graph_g2o_lie_algebra.cpp

#include <iostream>
#include <fstream>
#include <string>
#include <Eigen/Core>

#include <g2o/core/base_vertex.h>
#include <g2o/core/base_binary_edge.h>
#include <g2o/core/block_solver.h>
#include <g2o/core/optimization_algorithm_levenberg.h>
#include <g2o/solvers/eigen/linear_solver_eigen.h>

#include <sophus/se3.hpp>

using namespace std;
using namespace Eigen;
using Sophus::SE3d;
using Sophus::SO3d;

/************************************************
 * This program demonstrates how to use g2o solver for pose graph optimization
 * sphere.g2o is an artificially generated Pose graph, let's optimize it.
 * Although the entire graph can be read directly through the load function, we still implement the reading code ourselves in order to gain a deeper understanding.
 * This section uses Lie algebra to express the pose graph, and the nodes and edges are customized.
 * **********************************************/

typedef Matrix<double, 6, 6> Matrix6d;

// Find the approximation of J_R^{-1} given the error
Matrix6d JRInv(const SE3d &e) {
    Matrix6d J;
    J.block(0, 0, 3, 3) = SO3d::hat(e.so3().log());
    J.block(0, 3, 3, 3) = SO3d::hat(e.translation());
    J.block(3, 0, 3, 3) = Matrix3d::Zero(3, 3);
    J.block(3, 3, 3, 3) = SO3d::hat(e.so3().log());
    // J = J * 0.5 + Matrix6d::Identity();
    J = Matrix6d::Identity();    // try Identity if you want
    return J;
}

// Lie algebra vertex
typedef Matrix<double, 6, 1> Vector6d;

class VertexSE3LieAlgebra : public g2o::BaseVertex<6, SE3d> {
public:
    EIGEN_MAKE_ALIGNED_OPERATOR_NEW

    virtual bool read(istream &is) override {
        double data[7];
        for (int i = 0; i < 7; i++)
            is >> data[i];
        setEstimate(SE3d(
            Quaterniond(data[6], data[3], data[4], data[5]),
            Vector3d(data[0], data[1], data[2])
        ));
        return true;
    }

    virtual bool write(ostream &os) const override {
        os << id() << " ";
        Quaterniond q = _estimate.unit_quaternion();
        os << _estimate.translation().transpose() << " ";
        os << q.coeffs()[0] << " " << q.coeffs()[1] << " " << q.coeffs()[2] << " " << q.coeffs()[3] << endl;
        return true;
    }

    virtual void setToOriginImpl() override {
        _estimate = SE3d();
    }

    // left multiplier update
    virtual void oplusImpl(const double *update) override {
        Vector6d upd;
        upd << update[0], update[1], update[2], update[3], update[4], update[5];
        _estimate = SE3d::exp(upd) * _estimate;
    }
};

// Edge between two Lie algebra nodes
class EdgeSE3LieAlgebra : public g2o::BaseBinaryEdge<6, SE3d, VertexSE3LieAlgebra, VertexSE3LieAlgebra> {
public:
    EIGEN_MAKE_ALIGNED_OPERATOR_NEW

    virtual bool read(istream &is) override {
        double data[7];
        for (int i = 0; i < 7; i++)
            is >> data[i];
        Quaterniond q(data[6], data[3], data[4], data[5]);
        q.normalize();
        setMeasurement(SE3d(q, Vector3d(data[0], data[1], data[2])));
        for (int i = 0; i < information().rows() && is.good(); i++)
            for (int j = i; j < information().cols() && is.good(); j++) {
                is >> information()(i, j);
                if (i != j)
                    information()(j, i) = information()(i, j);
            }
        return true;
    }

    virtual bool write(ostream &os) const override {
        VertexSE3LieAlgebra *v1 = static_cast<VertexSE3LieAlgebra *> (_vertices[0]);
        VertexSE3LieAlgebra *v2 = static_cast<VertexSE3LieAlgebra *> (_vertices[1]);
        os << v1->id() << " " << v2->id() << " ";
        SE3d m = _measurement;
        Eigen::Quaterniond q = m.unit_quaternion();
        os << m.translation().transpose() << " ";
        os << q.coeffs()[0] << " " << q.coeffs()[1] << " " << q.coeffs()[2] << " " << q.coeffs()[3] << " ";

        // information matrix
        for (int i = 0; i < information().rows(); i++)
            for (int j = i; j < information().cols(); j++) {
                os << information()(i, j) << " ";
            }
        os << endl;
        return true;
    }

    // The error calculation is consistent with the derivation in the book
    virtual void computeError() override {
        SE3d v1 = (static_cast<VertexSE3LieAlgebra *> (_vertices[0]))->estimate();
        SE3d v2 = (static_cast<VertexSE3LieAlgebra *> (_vertices[1]))->estimate();
        _error = (_measurement.inverse() * v1.inverse() * v2).log();
    }

    // Jacobian calculation
    virtual void linearizeOplus() override {
        SE3d v1 = (static_cast<VertexSE3LieAlgebra *> (_vertices[0]))->estimate();
        SE3d v2 = (static_cast<VertexSE3LieAlgebra *> (_vertices[1]))->estimate();
        Matrix6d J = JRInv(SE3d::exp(_error));
        // Try to approximate J to I?
        _jacobianOplusXi = -J * v2.inverse().Adj();
        _jacobianOplusXj = J * v2.inverse().Adj();
    }
};

int main(int argc, char **argv) {
    if (argc != 2) {
        cout << "Usage: pose_graph_g2o_SE3_lie sphere.g2o" << endl;
        return 1;
    }
    ifstream fin(argv[1]);
    if (!fin) {
        cout << "file " << argv[1] << " does not exist." << endl;
        return 1;
    }

// *****************************************************************
    // Build graph optimization, first set up g2o
    typedef g2o::BlockSolver<g2o::BlockSolverTraits<6, 6>> BlockSolverType;
    // pose is 6, landmark is 6
    // The optimization variable dimension of each error term is 6, and the error value dimension is 6

    typedef g2o::LinearSolverEigen<BlockSolverType::PoseMatrixType> LinearSolverType; // Linear solver type
    // Gradient descent method, you can choose from GN, LM, DogLeg
    auto solver = new g2o::OptimizationAlgorithmLevenberg(
        std::make_unique<BlockSolverType>(std::make_unique<LinearSolverType>()));
    g2o::SparseOptimizer optimizer;   // graphical model
    optimizer.setAlgorithm(solver);   // Set up solver
    optimizer.setVerbose(true);       // Turn on debug output

    int vertexCnt = 0, edgeCnt = 0; // Number of vertices and edges

    vector<VertexSE3LieAlgebra *> vectices;
    vector<EdgeSE3LieAlgebra *> edges;
    while (!fin.eof()) {
        string name;
        fin >> name;
        if (name == "VERTEX_SE3:QUAT") {
            // vertex
            VertexSE3LieAlgebra *v = new VertexSE3LieAlgebra();
            int index = 0;
            fin >> index;
            v->setId(index);
            v->read(fin);
            optimizer.addVertex(v);
            vertexCnt++;
            vectices.push_back(v);
            if (index == 0)
                v->setFixed(true);
        } else if (name == "EDGE_SE3:QUAT") {
            // SE3-SE3 side
            EdgeSE3LieAlgebra *e = new EdgeSE3LieAlgebra();
            int idx1, idx2;     // Two related vertices
            fin >> idx1 >> idx2;
            e->setId(edgeCnt++);
            e->setVertex(0, optimizer.vertices()[idx1]);
            e->setVertex(1, optimizer.vertices()[idx2]);
            e->read(fin);
            optimizer.addEdge(e);
            edges.push_back(e);
        }
        if (!fin.good()) break;
    }

    cout << "read total " << vertexCnt << " vertices, " << edgeCnt << " edges." << endl;

    cout << "optimizing ..." << endl;
    optimizer.initializeOptimization();
    optimizer.optimize(30);

    cout << "saving optimization results ..." << endl;

    // Because custom vertices are used and not registered with g2o, save yourself here to implement it.
    //Disguise as SE3 vertices and edges so that g2o_viewer can recognize them
    ofstream fout("result_lie.g2o");
    for (VertexSE3LieAlgebra *v:vectices) {
        fout << "VERTEX_SE3:QUAT ";
        v->write(fout);
    }
    for (EdgeSE3LieAlgebra *e:edges) {
        fout << "EDGE_SE3:QUAT ";
        e->write(fout);
    }
    fout.close();
    return 0;
}

Overwriting pose_graph_g2o_lie_algebra.cpp


In [ ]:
%%writefile CMakeLists.txt
cmake_minimum_required(VERSION 3.22)
project(vo1)

set(CMAKE_BUILD_TYPE "Release")
set(CMAKE_CXX_FLAGS "-std=c++17 -O2")

# Eigen
find_package(Eigen3 REQUIRED)
include_directories(${EIGEN3_INCLUDE_DIR})

# OpenCV
find_package(OpenCV REQUIRED)
include_directories(${OpenCV_INCLUDE_DIRS})

# Sophus
find_package(Sophus REQUIRED)
include_directories(${Sophus_INCLUDE_DIRS})

# g2o
find_package(g2o REQUIRED)
include_directories(${g2o_INCLUDE_DIRS})


add_executable(pose_graph_g2o_lie pose_graph_g2o_lie_algebra.cpp)
target_link_libraries(pose_graph_g2o_lie
         g2o_core g2o_stuff g2o_types_sba g2o_types_slam3d g2o_solver_dense
        Sophus::Sophus
        ${CHOLMOD_LIBRARIES}
        ${Sophus_LIBRARIES}
        )


Overwriting CMakeLists.txt


In [ ]:
!cp "/content/drive/MyDrive/slambook_data/ch10/sphere.g2o" /content/MyLiePGExample

In [ ]:
!mkdir myBuild
%cd myBuild/

mkdir: cannot create directory ‘myBuild’: File exists
/content/MyLiePGExample/myBuild


In [ ]:
!cmake ..

-- Configuring done (0.0s)
-- Generating done (0.0s)
-- Build files have been written to: /content/MyLiePGExample/myBuild


In [ ]:
!make pose_graph_g2o_lie

[ 50%] Building CXX object CMakeFiles/pose_graph_g2o_lie.dir/pose_graph_g2o_lie_algebra.cpp.o
[100%] Linking CXX executable pose_graph_g2o_lie
[100%] Built target pose_graph_g2o_lie


In [ ]:
! ./pose_graph_g2o_lie

./pose_graph_g2o_lie: error while loading shared libraries: libg2o_core.so.0.2: cannot open shared object file: No such file or directory


In [ ]:
! LD_LIBRARY_PATH=/usr/local/lib:$LD_LIBRARY_PATH ./pose_graph_g2o_lie /content/MyLiePGExample/sphere.g2o

read total 2500 vertices, 9799 edges.
optimizing ...
iteration= 0	 chi2= 674837160.579974	 time= 0.50803	 cumTime= 0.50803	 edges= 9799	 schur= 0	 lambda= 6658.554263	 levenbergIter= 1
iteration= 1	 chi2= 234706314.970484	 time= 0.467333	 cumTime= 0.975363	 edges= 9799	 schur= 0	 lambda= 2219.518088	 levenbergIter= 1
iteration= 2	 chi2= 142146174.348537	 time= 0.461921	 cumTime= 1.43728	 edges= 9799	 schur= 0	 lambda= 739.839363	 levenbergIter= 1
iteration= 3	 chi2= 83834595.145595	 time= 0.474523	 cumTime= 1.91181	 edges= 9799	 schur= 0	 lambda= 246.613121	 levenbergIter= 1
iteration= 4	 chi2= 41878079.903257	 time= 0.46561	 cumTime= 2.37742	 edges= 9799	 schur= 0	 lambda= 82.204374	 levenbergIter= 1
iteration= 5	 chi2= 16598628.119946	 time= 0.496862	 cumTime= 2.87428	 edges= 9799	 schur= 0	 lambda= 27.401458	 levenbergIter= 1
iteration= 6	 chi2= 6137666.739406	 time= 0.458997	 cumTime= 3.33328	 edges= 9799	 schur= 0	 lambda= 9.133819	 levenbergIter= 1
iteration= 7	 chi2= 2182986.250